# Genius API Sample Checks
Use this notebook to quickly search Genius and fetch lyrics for manual spot checks.

In [27]:
import os
import lyricsgenius

GENIUS_ACCESS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN", "")
if not GENIUS_ACCESS_TOKEN:
    raise ValueError("Set GENIUS_ACCESS_TOKEN in your environment or directly in this cell.")

genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=15,
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True,
 )
genius.verbose = False

In [28]:
def fetch_lyrics_genius(client, title: str, artist: str) -> str:
    try:
        hit = client.search_song(title=title, artist=artist)
        if hit and hit.lyrics:
            return hit.lyrics.strip()
    except Exception as e:
        print(f"[warn] {title!r} by {artist!r}: {e}")
    return ""

In [49]:
# STEP 1: Search Title using spotify_uri from titles.csv
spotify_uri = "1OHj2WRXOR9XdCV6PuptXv"
titles_df = pd.read_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/titles/2026/03/05/titles.csv"))
matching_row = titles_df[titles_df["spotify_uri"] == spotify_uri]
song_title = matching_row["title"].iloc[0] if not matching_row.empty else None
song_artist = matching_row["artist"].iloc[0] if not matching_row.empty else None
print(f"\nSpotify URI: {spotify_uri}")
print(f"Title: {song_title}")
print(f"Artist: {song_artist}")


Spotify URI: 1OHj2WRXOR9XdCV6PuptXv
Title: Amarte más no pude - En vivo
Artist: Luister La Voz


In [ ]:
# STEP 1: Search Lyrics using Title
song_title = "Rein me in"
song_artist = ""

lyrics = fetch_lyrics_genius(genius, song_title, song_artist)

print("\nLyrics snippet:\n")
print((lyrics or "<no lyrics found>")[:1200])
# print artist and title from genius search result



Lyrics snippet:

Eh-yaoh, Cypher volumen uno, hermano, estamo' acá con Saje, estamo' con Kelo, con Urbanse, con el Kundo13, con Santoz Interestelar, Brapos Caliope Fam, MPDhela
Esta mierda va a sonar gorda, hermano, A-C-R-U Román quien lo habla, Cypher Volumen Uno
Let's go


Okey, yeah

Pista minimalista, mi rival imita lírica mixta
Cínica, rítmica, mítica vista
Vi mi graffiti en mi empírica city, tapan puro lápiz y lata
Duro cáliz y la partícula circular mi curar
Para mí es Hippy Happa y solo te ofrezco una pizca
Quiero beber de la fuente, ser el gestor de una vida consciente
El narrador de la historia que invente, de la vertiente esta flor ha caído
Pasan las horas y meses de olvido, como la cuido yo si pude ver
Que hasta el pantano puede florecer según el abrigo que brinde la gente
Y es difícil de creer, pensar en saltar, luego retroceder
Si sé que mañana tendré que volver al mismo camino que me hizo más fuerte
Es relativa la muerte si no sé qué significa estar vivo
No basta con esc

In [51]:
# STEP 2: Get spotify_uri from titles.csv by title and artist
from pathlib import Path
import pandas as pd

titles_df = pd.read_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/titles/2026/03/05/titles.csv"))
matching_row = titles_df[
    titles_df["title"].str.contains(song_title, case=False, na=False) &
    (
        titles_df["artist"].str.contains(song_artist, case=False, na=False)
        if song_artist != "" else True
    )
]

song_uri = matching_row["spotify_uri"].iloc[0] if not matching_row.empty else None
song_title = matching_row["title"].iloc[0] if not matching_row.empty else song_title
song_artist = matching_row["artist"].iloc[0] if not matching_row.empty else song_artist

print(f"\nSpotify URI for '{song_title}' by '{song_artist}': {song_uri or '<not found>'}")


Spotify URI for 'Amarte más no pude - En vivo' by 'Luister La Voz': 1OHj2WRXOR9XdCV6PuptXv


In [52]:
# STEP 3: Update titles.csv with song_title and song_artist for the row with the matching spotify_uri (if found)
if song_uri:
    titles_df.loc[titles_df["spotify_uri"] == song_uri, ["title", "artist"]] = [song_title, song_artist]
    titles_df.to_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/titles/2026/03/05/titles.csv"), index=False)
    print(f"Updated titles.csv with title '{song_title}' and artist '{song_artist}' for spotify_uri '{song_uri}'.")
else:
    print(f"No matching spotify_uri found for title '{song_title}' and artist '{song_artist}'. No updates made to titles.csv.")

Updated titles.csv with title 'Amarte más no pude - En vivo' and artist 'Luister La Voz' for spotify_uri '1OHj2WRXOR9XdCV6PuptXv'.


In [53]:
# STEP 4: Update lyrics.csv with lyrics for the matching spotify_uri
lyrics_df = pd.read_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics.csv"))
if song_uri and lyrics:
    lyrics_df.loc[lyrics_df["spotify_uri"] == song_uri, "lyrics"] = lyrics
    lyrics_df.to_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics.csv"), index=False)
    print(f"Updated lyrics.csv with lyrics for spotify_uri '{song_uri}'.")
else:
    print(f"No matching spotify_uri or lyrics found for title '{song_title}' and artist '{song_artist}'. No updates made to lyrics.csv.")

Updated lyrics.csv with lyrics for spotify_uri '1OHj2WRXOR9XdCV6PuptXv'.
